**Introductory and intermediate computing for Data Science [Barcelona School of Economics]**

`Instructor:` Maxim Fedotov  
`Program:` M.Sc. in Data Science Methodology

# Class 5: JSON and pandas

## JSON

JSON is a convenient data format which has a wide range of application from representing tabular data to storing web-responses. There is a specific syntax for this data format. A simple example is presented below:

```{JSON}
{"holder": "Foo", 
 "account_no": 1337, 
 "balance": 100500}
```

However, it is not the only possible structure. It can be also: a dictionary where values are lists, a nested dictionary, a list of dictionaries and similar structures. Note that in JSON format you must not use single quotes `'`. 

In particular, these formats can be easily converted to pythonic objects. Package `json` gives us this functionality.

In [ ]:
import json

Let's have a look at how we can convert a simple string written in a JSON format into a pythonic object:

In [ ]:
json_str = '{"variable1": [4.45654794846065, 4.278310755227896, 3.009643269934777], "variable2": [3, 3, 3]}'
data = json.loads(json_str)
data

The `json.loads(...)` function "loads" a string, i.e. transforms it to a dictionary in this case. 

We can also write pythonic objects to text files with the "json" extension. We can use a *context manager* which allows us to open files and work with them in different modes. To define a context manager, we use the keyword `with` together with the `open(...)` function. In the `open(...)` function we specify a path to a file, and a mode in which we want to open it, e.g. "w" for writing (but truncating an existing file at first), "a" to append content, "r" to read and others.

In [ ]:
with open("data.json", "w") as file:
    json.dump(data, file)

You could do it in another way using just the `open(...)` function. But you **have to** close a file after use, otherwise you can encounter some problems.

In [ ]:
file = open("data.json", "w")
json.dump(data, file)
file.close()

So, use of a *context manager* is preferable.

You can also read files with json.

In [ ]:
with open("data.json", "r") as file:
    data_new = json.load(file)

In [ ]:
data_new

Just as a side comment, you can also read any text file line by line:

In [ ]:
with open("data.json", "r") as file:  # our file consists of only one line, but you get the idea
    data_txt = []
    for line in file:
        data_txt.append(line)

You could also use functions like `file.readline(...)`, `file.readlines(...)`, `file.seek(...)` and others to work with files. Feel free to find out what they do.

One format you will meet often is *JSON Lines*: a text file where each line is a complete, separate JSON object. To read it, you combine the two things above — read the file line by line, and call `json.loads(...)` on each line.

## Building a dataset to work with

Before we look at the `pandas` package that introduces many useful tools for data engineering and analysis, let's build a small dataset to use with it. We simulate it rather than download one, so that we know exactly what is in it.

We will generate three variables:

* `age` — integer, the age of a person
* `female` — binary indicator of sex
* `occupation` — one of a few job titles, drawn with equal probability

and then a `wage` that depends on age and occupation, plus some noise. If you would like to see a more careful treatment of simulating from a linear model, including recovering the coefficients afterwards, see the numpy notebook.

In [ ]:
import numpy as np

np.random.seed(1337)   # setting the seed for reproducibility
n = 100

occupations = ["data analyst", "data engineer", "software engineer", "tester"]

X_dict = {
    "age": np.random.randint(18, 65, size=n),
    "female": np.random.binomial(1, 0.5, size=n),
    "occupation": np.random.choice(occupations, size=n),
}

{key: value[:5] for key, value in X_dict.items()}   # a peek at the first five entries for each variable

Now the outcome. Wage grows with age, differs a little by occupation, and has some random noise on top:

In [ ]:
occupation_effect = {
    "data analyst": 0.0,
    "data engineer": -5,
    "software engineer": -5,
    "tester": -6,
}

y = (
    0.5 * X_dict["age"]
    + np.array([occupation_effect[job] for job in X_dict["occupation"]])
    + np.random.normal(0, 1, size=n)
)

y[:10]

## Pandas

One particularly helpful package when you work with tabular data is `pandas`. Let's import it:

In [ ]:
import pandas as pd

The package defines `pandas.DataFrame` class, so we can create dataframes. To create a dataframe, you can call a class constructor `pandas.DataFrame(...)` and pass some data into it. In fact, there are different ways to create dataframes. You can explore them in the corresponding docstring by executing `help(pandas.DataFrame)`.

Now let's pass the data we had in a dictionary format to the pandas DataFrame constructor to see what it gives us.

In [ ]:
data = pd.DataFrame(X_dict)
data.head(10)

Dataframes also have useful methods. You can see how we use `.head(...)` method in the above cell. There are also other methods  and properties of dataframes that help us explore their contents, like:

In [ ]:
print('Explore the "tail":', data.tail(), sep='\n')
print('Explore what columns the dataframe has:', data.columns)  # notice the specific type of the output
print('Explore the shape and size:', str(data.shape) + ", " + str(data.size))

You can access a column of a dataframe using `[]` or call it as a property like that:

In [ ]:
print('Access the "age" column:', data["age"], sep='\n', end='\n\n')
print('Access it as a property of the dataset:', data.age, sep='\n')

But note, that it works only with existing columns. To create a new column, simply provide a name in `[]` right after the name of the dataframe and assign it a specific iterable of a valid size.

In [ ]:
data['wage'] = y  # this is the output we have simulated before
data.head()

Note that you can pick specific fields or slices from a dataframe using the properties `.loc[...]` and `.iloc[...]`. The former selects fields based on index *names*, i.e. the names of rows and columns (even when those names are numbers). The latter uses positional indices, like `.iloc[1:3, 3]`, which takes the elements at the intersection of the 2nd and 3rd rows with the 4th column.

Be careful with slices: **for `.loc[...]` both start and end are included** (unlike usual Python slices), whereas **for `.iloc[...]` the start is included and the end is excluded**.

The following two lines give the same result.

In [ ]:
print(data.iloc[1:10, 0:3], end="\n\n")
print(data.loc[1:9, ["age", "female", "occupation"]])

It is possible (and quite useful) to use boolean iterables for subsetting. For example:

In [ ]:
print(
      "We can get all the observations "
      "with age between 22 and 40 and wage greater than 20:", 
      data.loc[data.age.between(22, 40) & data.wage.gt(20), :], 
      sep='\n', 
      end='\n\n'
)

print(
      "We can get some columns for all the observations "
      "that correspond to data engineers and analysts", 
      data.loc[
            data.occupation.eq("data engineer") | (data.occupation == "data analyst"),  # parentheses are crucial
            ["age", "wage", "occupation"]
      ], 
      sep='\n'
)

Note the use of element-wise logical operations `&` and `|`. You might also need `~` which is element-wise negation.

We can also try to find out what is the type of a column:

In [ ]:
type(data['wage'])

That is a pandas series object. These are created to store a sequence of elements, and they provide mathematical and statistical methods to explore this data (so as pandas dataframes have). 

In [ ]:
print("Get the maximum wage:", data['wage'].max())
print("Get the mean wage:", data['wage'].mean())
print("Get the min wage:", data['wage'].min())

Note that you should always try to vectorize mathematical operations with series. The pandas objects allow for that; in fact, proper vectorization is much more efficient than iterating over elements. As one simple example, let's compute the square of age (note: the dependence of wage on age is often claimed to be quadratic in labour economics):

In [ ]:
data["age_sq"] = data.age**2
data["wage_log"] = np.log(data.wage)

Our `occupation` column holds text, but most statistical and machine-learning tools need numbers. The usual way to handle a variable taking one of $v$ discrete values is *one-hot encoding*: replace it with $v$ indicator columns, each 0 or 1.

Usually one of the indicators is dropped and treated as a baseline. If all the remaining $v - 1$ indicators are zero, the baseline value is implied, so the dropped column carries no extra information — and keeping it would introduce perfect *multicollinearity* into a regression.

pandas does this in one call:

In [ ]:
occupation_dummies = pd.get_dummies(data.occupation, prefix="occ", drop_first=True)
occupation_dummies.head()

`drop_first=True` is what leaves out the baseline category. To attach these columns to the dataframe, use `pd.concat` along the column axis:

In [ ]:
data_encoded = pd.concat([data, occupation_dummies], axis=1)
data_encoded.head()

You can use `.agg(...)` method to apply different aggregating functions to different columns like that:

In [ ]:
data.agg({"age": "mean", "wage": "max"})

This is a powerful tool which you will most likely use later too.

Several technical, but useful methods would be `.replace(...)`, `.drop(...)`, and `.rename(...)`.

Let's suppose that we want to replace some values to NaNs (just as example):

In [ ]:
data['occupation'] = data.occupation.replace(["tester", "software engineer"], np.nan)

And now we can drop all the rows that contain NaNs:

In [ ]:
data.dropna(inplace=True)

You can inspect the index of the new dataframe using its `.index` property:

In [ ]:
data.index

You can see that after we dropped some rows, the indices did not change — there are now gaps. If we then try to slice the dataframe with `.loc[]`, we can run into problems (whereas `.iloc[]` would still work). So it is recommended to reset the indices after introducing changes to the original dataset, if you do not need the old ones. `drop=True` discards the old index rather than keeping it as a new column.

In [ ]:
data.reset_index(drop=True, inplace=True)
data.index

You can drop columns (or rows) using the following method:

In [ ]:
data.drop(columns=['wage'], inplace=True)

We can also rename columns:

In [ ]:
data.rename(columns={'age': 'experience', 'age_sq': 'experience_sq'}, inplace=True)

Note that you could also rename your row indices.

It is very important to pay attention to the `inplace` arguments in pandas methods you use, because it defines whether a method mutates an initial object or returns a new one.

In addition, whenever you want to create a new dataframe from an existing one, consider using `.copy()` method that returns a copy of a dataframe.

In [ ]:
data_new = data.loc[0:100, ["wage_log", "experience", "occupation"]].copy()

The problem is in how Python handles assignments. In this case a new variable would refer to an old one, and changes to the new one can mutate the original object.

You can save your data to a file using the `.to_...` methods. `index=False` stops pandas writing the row index out as an extra column, which is almost always what you want for a CSV:

In [ ]:
data.to_csv('class_5_data.csv', index=False)

To read this data later, use functions of form `pandas.read_...`:

In [ ]:
data_read = pd.read_csv('class_5_data.csv')
data_read.head()

## A quick look at plotting

DataFrames and Series have their own `.plot` methods, built on top of `matplotlib`, so you do not need to leave pandas to get a quick look at your data.

In [ ]:
data_read.plot.scatter(x="experience", y="wage_log", figsize=(5, 3.5))

You can also plot the result of an aggregation directly. Here we compute the mean and standard deviation together with `.agg(...)`, then pass the standard deviation as `yerr` to get error bars.

In [ ]:
wage_by_occupation = data_read.groupby("occupation")["wage_log"].agg(["mean", "std"])
wage_by_occupation["mean"].plot.bar(yerr=wage_by_occupation["std"], capsize=4, figsize=(5, 3.5))

There is a lot more in `pandas` — `.plot.hist()`, `.plot.box()`, `.plot.line()`, and options to control almost every visual detail. For anything beyond a quick look, you will also want `matplotlib` directly, which gives you full control over a figure, and `seaborn`, which is built on top of `matplotlib` and specialises in statistical plots — both are worth exploring on your own.

## Afterword

Of course, pandas is a massive package. The information in this notebook provides you with basic objects, their properties and methods in pandas, so you can get acquainted with it and build on that knowledge further.

Some of the topics that we have not covered here but can be of your interest in Pandas (so feel free to check them out):

* multi-indexing
* plotting with pandas (and in general with `matplotlib`)
* mapping functions to series (but always try to vectorize computations, then you won't need it often), one of the use cases is "renaming" values (use dictionary).
* datetime format

For sure, you should not stop on that. Your journey just begins. Good luck!